# Suppressing instability on a Vlasov-Poisson system - Two-Stream (NVIDIA Warp backend)

This notebook is a Warp port of the Two-Stream optimization example in the original `vlasov-poisson` repository.  The forward solver runs on Warp kernels (CUDA when available, CPU otherwise); gradients flow through a custom `jax.custom_vjp` adjoint so Optax loops are unchanged.


## Setup


In [ ]:
# Colab setup - skip if running locally with the package already installed.
import importlib, subprocess, sys

def _ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg,
        ])

for pkg, pip_name in [
    ('warp', 'warp-lang'),
    ('jax', 'jax'),
    ('optax', 'optax'),
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
]:
    _ensure(pkg, pip_name)

# Install the warp_vp_solver package itself.
try:
    import warp_vp_solver  # noqa: F401
except ImportError:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/Forgotten/warp_vp_solver.git@main',
    ])

import warp as wp
wp.init()
print(f'Warp device: {wp.get_preferred_device()}')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import optax
from tqdm import trange

from warp_vp_solver import (
    make_mesh,
    WarpVlasovPoissonSolver,
    external_electric_field,
    make_cost_function_kl,
    make_cost_function_ee,
    plot_inital_solve,
    plot_results_TS,
    plot_results_BoT,
)

jax.config.update('jax_enable_x64', True)


## Mesh and equilibrium distribution


In [ ]:
nx, nv = 128, 128
L = 10.0 * np.pi
LV = 6.0
dt = 0.1
t_final = 30.0
t_values = np.linspace(0.0, t_final, int(t_final / dt))
k_0 = 0.2

mesh = make_mesh(L, LV, nx, nv)
mu1 = 2.4
f_eq = (np.exp(-0.5 * (mesh.V - mu1) ** 2)
        + np.exp(-0.5 * (mesh.V + mu1) ** 2)) / (2.0 * np.sqrt(2.0 * np.pi))
epsilon1 = 0.001
f_iv = (1.0 + epsilon1 * np.cos(k_0 * mesh.X)) * f_eq


## Build the Warp solver


In [ ]:
solver = WarpVlasovPoissonSolver(mesh=mesh, dt=dt, f_eq=f_eq)
solver_jit = solver.run_forward_jax  # custom_vjp wrapper


## Forward run with H=0 and a hand-tuned initial H


In [ ]:
ak1 = jnp.array([[-6.31537e-9], [-0.00128741]])  # shape (2, N)
H1 = external_electric_field(ak1, mesh, k_0)

f1, fh1, Eh1, ee1 = solver_jit(jnp.asarray(f_iv), 0.0 * H1, t_final)
f2, fh2, Eh2, ee2 = solver_jit(jnp.asarray(f_iv), H1, t_final)


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(28, 6))
plot_inital_solve(fig, axs, f_eq, f1, ee1, f2, ee2, mesh, t_values, sci=False)
plt.show()


## Optimize the external field with Optax


In [ ]:
k_total = 2
ak_init = jax.random.uniform(
    jax.random.key(888), (2, k_total), minval=-0.003, maxval=0.001,
)

cost_fn_kl = make_cost_function_kl(
    solver, solver_jit, jnp.asarray(f_iv), k_0, t_final,
)

value_and_grad = jax.value_and_grad(cost_fn_kl)

opt = optax.adam(learning_rate=1e-3)
params = ak_init
opt_state = opt.init(params)

objective_history = []
for step in trange(20):
    val, grad = value_and_grad(params)
    updates, opt_state = opt.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    objective_history.append(float(val))

objective_history = np.asarray(objective_history)
print('Final KL:', objective_history[-1])


## Final solution and convergence


In [ ]:
H_opt = external_electric_field(params, mesh, k_0)
f_opt, fh_opt, Eh_opt, ee_opt = solver_jit(
    jnp.asarray(f_iv), H_opt, t_final,
)

fig, axs = plt.subplots(1, 4, figsize=(28, 6))
plot_results_TS(
    fig, axs, f_opt, Eh_opt, np.asarray(H_opt), ee_opt,
    objective_history, t_values, mesh,
)
plt.show()
